# Notebook 21 — Project Training Pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/21_project_training_pipeline.ipynb)

This notebook turns the capstone dataset into an experimental post-training pipeline for Castalia Mentor. The center of gravity is QLoRA supervised finetuning, but the notebook also prepares optional preference alignment, comparison runs, checkpoint policy, and failure analysis.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Runtime strategy**
- Understand **Step 1: Add helpers, flags, and artifact paths**
- Understand **Step 2: Load the capstone dataset from notebook 20**
- Understand **Step 3: Inspect the loaded records**


## What you will build

- a dataset loader that reads notebook 20 artifacts or falls back to a local mini corpus
- SFT-ready chat records and optional preference pairs
- configuration surfaces for model choice, LoRA scope, and run planning
- comparison runs for Colab-friendly QLoRA experiments
- checkpoint manifests, optional real training hooks, and failure buckets
- a handoff package for notebook 22 benchmarking and export work

## Runtime strategy

The notebook is designed to stay runnable in two modes. In planning mode, it simulates several runs so you can reason about trade-offs cheaply. In execution mode, you can flip the flags to load the base model and run a small real SFT or alignment stage on Colab.

In [ ]:
# --- Capstone training setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_from_disk
from peft import PeftModel
from transformers import TrainingArguments
from trl import SFTTrainer

try:
    from trl import DPOTrainer
except Exception:
    DPOTrainer = None

from unsloth import FastLanguageModel

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)
random.seed(21)
torch.manual_seed(21)

LOAD_BASE_MODEL = False
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"

model = None
tokenizer = None

if LOAD_BASE_MODEL:
    from google.colab import drive

    drive.mount('/content/drive')
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        cache_dir=CACHE_DIR,
    )
    tokenizer.padding_side = "right"
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

print("Training setup ready")
print("Base model loaded:", model is not None)
print("DPO trainer available:", DPOTrainer is not None)

## Step 1: Add helpers, flags, and artifact paths

These helpers support both the cheap planning path and the optional live training path.

In [ ]:
PROJECT_NAME = "Castalia Mentor"
ARTIFACT_DIR = Path("artifacts") / "notebook_21_project_training_pipeline"
RUN_ROOT = ARTIFACT_DIR / "runs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ACTUAL_SFT = False
RUN_ALIGNMENT_STAGE = False

def stable_hash(value, length=12):
    import hashlib

    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:length]

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def native_value(value):
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return value
    return value

print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2: Load the capstone dataset from notebook 20

If the versioned dataset artifacts already exist, we load them directly. If not, the notebook falls back to a tiny local corpus so the orchestration cells still run end to end.

In [ ]:
DATASET_VERSION = "fallback_dataset"
dataset_manifest = {"project": PROJECT_NAME, "dataset_version": DATASET_VERSION, "source": "fallback"}

def build_fallback_dataset():
    base_rows = {
        "train": [
            {
                "example_id": "fallback_train_1",
                "dataset_version": DATASET_VERSION,
                "source_track": "finetuning",
                "scenario_family": "qlora_recipe",
                "mentor_intent": "teach first run scoping",
                "difficulty": "foundation",
                "origin": "seed",
                "rubric_tags": ["qlora", "colab", "first-run"],
                "messages": [
                    {"role": "system", "content": "You are Castalia Mentor, a domain-specialized mentor. Teach clearly and state uncertainty honestly."},
                    {"role": "user", "content": "What is a safe first QLoRA recipe on a T4?"},
                    {"role": "assistant", "content": "Use a small open instruct model, 4-bit loading, a short held-out split, and a brief first run before wider ablations."},
                ],
            },
            {
                "example_id": "fallback_train_2",
                "dataset_version": DATASET_VERSION,
                "source_track": "evals",
                "scenario_family": "uncertainty_boundary",
                "mentor_intent": "teach calibration",
                "difficulty": "core",
                "origin": "seed",
                "rubric_tags": ["calibration", "safety"],
                "messages": [
                    {"role": "system", "content": "You are Castalia Mentor, a domain-specialized mentor. Teach clearly and state uncertainty honestly."},
                    {"role": "user", "content": "What should the mentor say when it is unsure?"},
                    {"role": "assistant", "content": "State the uncertainty plainly, keep the reliable part of the answer, and suggest the next source or experiment that would resolve the gap."},
                ],
            },
            {
                "example_id": "fallback_train_3",
                "dataset_version": DATASET_VERSION,
                "source_track": "agents",
                "scenario_family": "tool_retry",
                "mentor_intent": "teach tool recovery",
                "difficulty": "stretch",
                "origin": "seed",
                "rubric_tags": ["agents", "tool-use", "debugging"],
                "messages": [
                    {"role": "system", "content": "You are Castalia Mentor, a domain-specialized mentor. Teach clearly and state uncertainty honestly."},
                    {"role": "user", "content": "How should a student recover from a timed-out tool call?"},
                    {"role": "assistant", "content": "Name the failed step, preserve the partial observations, retry with a narrower query, and escalate only after the second failure."},
                ],
            },
        ],
        "validation": [
            {
                "example_id": "fallback_val_1",
                "dataset_version": DATASET_VERSION,
                "source_track": "rag",
                "scenario_family": "grounding_gap",
                "mentor_intent": "teach evidence use",
                "difficulty": "core",
                "origin": "seed",
                "rubric_tags": ["rag", "grounding", "uncertainty"],
                "messages": [
                    {"role": "system", "content": "You are Castalia Mentor, a domain-specialized mentor. Teach clearly and state uncertainty honestly."},
                    {"role": "user", "content": "What should the mentor do when retrieved notes disagree?"},
                    {"role": "assistant", "content": "State what the notes agree on, flag the contradiction, and ask for a stronger source before claiming certainty."},
                ],
            },
        ],
        "test": [
            {
                "example_id": "fallback_test_1",
                "dataset_version": DATASET_VERSION,
                "source_track": "finetuning",
                "scenario_family": "export_handoff",
                "mentor_intent": "teach deployment trade-offs",
                "difficulty": "core",
                "origin": "seed",
                "rubric_tags": ["deployment", "benchmarking"],
                "messages": [
                    {"role": "system", "content": "You are Castalia Mentor, a domain-specialized mentor. Teach clearly and state uncertainty honestly."},
                    {"role": "user", "content": "When should I keep an adapter separate instead of merging it?"},
                    {"role": "assistant", "content": "Keep it separate when rollback and variant reuse matter. Merge when deployment wants one artifact and the benchmark evidence is already strong."},
                ],
            },
        ],
    }

    for split_name, rows in base_rows.items():
        for row in rows:
            row["text"] = "\n\n".join("<|" + message["role"] + "|>\n" + message["content"] for message in row["messages"])
    return DatasetDict({split_name: Dataset.from_list(rows) for split_name, rows in base_rows.items()})

def load_project_dataset():
    global DATASET_VERSION
    version_root = Path("artifacts") / "notebook_20_project_dataset_pipeline" / "versions"
    if not version_root.exists():
        return build_fallback_dataset(), dataset_manifest

    version_dirs = sorted(
        [path_obj for path_obj in version_root.iterdir() if path_obj.is_dir()],
        key=lambda path_obj: path_obj.stat().st_mtime,
        reverse=True,
    )
    if not version_dirs:
        return build_fallback_dataset(), dataset_manifest

    latest_version_dir = version_dirs[0]
    manifest_path = latest_version_dir / "manifest.json"
    dataset_path = latest_version_dir / "hf_dataset"
    if manifest_path.exists():
        loaded_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    else:
        loaded_manifest = dataset_manifest

    DATASET_VERSION = loaded_manifest.get("dataset_version", latest_version_dir.name)
    if dataset_path.exists():
        return load_from_disk(str(dataset_path)), loaded_manifest

    return build_fallback_dataset(), loaded_manifest

project_dataset, dataset_manifest = load_project_dataset()
DATASET_VERSION = dataset_manifest.get("dataset_version", DATASET_VERSION)

display(pd.DataFrame(
    [
        {"split": split_name, "rows": len(project_dataset[split_name])}
        for split_name in project_dataset.keys()
    ]
))
display(pd.DataFrame([dataset_manifest]))

## Step 3: Inspect the loaded records

Before choosing training settings, verify that the records look like what the pipeline is supposed to teach.

In [ ]:
preview_rows = []
for split_name in project_dataset.keys():
    for row in project_dataset[split_name][: min(2, len(project_dataset[split_name]))]:
        preview_rows.append(
            {
                "split": split_name,
                "example_id": row["example_id"],
                "source_track": row["source_track"],
                "scenario_family": row["scenario_family"],
                "difficulty": row["difficulty"],
            }
        )

display(pd.DataFrame(preview_rows))
print(project_dataset)

## Step 4: Render SFT-ready chat text and preserve the prompt contract

If the dataset already includes text, we keep it. Otherwise we create a deterministic chat rendering that works even before the tokenizer is loaded.

In [ ]:
SYSTEM_PROMPT = "You are Castalia Mentor, a domain-specialized mentor. Teach clearly, stay evidence-aware, and state uncertainty honestly."

def ensure_messages(row):
    if "messages" in row and row["messages"]:
        return row["messages"]
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["response"]},
    ]

def render_chat_text(messages):
    if tokenizer is not None:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return "\n\n".join("<|" + message["role"] + "|>\n" + message["content"] for message in messages)

def normalize_split_for_sft(dataset_split):
    normalized_rows = []
    for row in dataset_split:
        messages = ensure_messages(row)
        normalized_rows.append(
            {
                **row,
                "messages": messages,
                "text": row["text"] if "text" in row and row["text"] else render_chat_text(messages),
            }
        )
    return Dataset.from_list(normalized_rows)

sft_splits = DatasetDict({split_name: normalize_split_for_sft(project_dataset[split_name]) for split_name in project_dataset.keys()})
display(pd.DataFrame(sft_splits["train"][:3])[["example_id", "scenario_family", "source_track"]])

## Step 5: Expose the main configuration surfaces

Good capstone work separates the knobs that matter: model recipe, adapter scope, and training loop policy.

In [ ]:
model_surface = pd.DataFrame(
    [
        {"surface": "model_name", "default": MODEL_NAME, "why_it_matters": "Determines base quality and VRAM footprint."},
        {"surface": "max_seq_length", "default": MAX_SEQ_LENGTH, "why_it_matters": "Controls context headroom and activation cost."},
        {"surface": "load_in_4bit", "default": True, "why_it_matters": "Keeps the base checkpoint Colab-friendly."},
    ]
)

adapter_surface = pd.DataFrame(
    [
        {"surface": "lora_rank", "default": 16, "why_it_matters": "More capacity can help but raises trainable parameter count."},
        {"surface": "lora_alpha", "default": 16, "why_it_matters": "Scales adapter updates."},
        {"surface": "lora_dropout", "default": 0.0, "why_it_matters": "Regularization knob for broader datasets."},
        {"surface": "target_modules", "default": "attention plus MLP projections", "why_it_matters": "Defines where the adapter can change behavior."},
    ]
)

training_surface = pd.DataFrame(
    [
        {"surface": "learning_rate", "default": 1e-4, "why_it_matters": "Too high destabilizes; too low wastes steps."},
        {"surface": "max_steps", "default": 36, "why_it_matters": "Keeps early runs cheap and debuggable."},
        {"surface": "packing", "default": True, "why_it_matters": "Improves throughput if formatting is already trustworthy."},
        {"surface": "evaluation_strategy", "default": "steps", "why_it_matters": "Lets validation catch regressions mid-run."},
    ]
)

display(model_surface)
display(adapter_surface)
display(training_surface)

## Step 6: Build a comparison grid for candidate QLoRA runs

The point of the grid is not brute force. It is to make the trade-offs explicit before any expensive training begins.

In [ ]:
run_grid = pd.DataFrame(
    [
        {
            "run_name": "t4_safe_sft",
            "hardware": "T4",
            "rank": 16,
            "learning_rate": 1.0e-4,
            "max_steps": 32,
            "seq_length": 1536,
            "packing": True,
            "alignment_stage": "none",
            "notes": "Conservative first release candidate.",
        },
        {
            "run_name": "t4_higher_rank",
            "hardware": "T4",
            "rank": 32,
            "learning_rate": 9.0e-5,
            "max_steps": 36,
            "seq_length": 1536,
            "packing": True,
            "alignment_stage": "none",
            "notes": "More adapter capacity on the same GPU budget.",
        },
        {
            "run_name": "t4_long_context",
            "hardware": "T4",
            "rank": 16,
            "learning_rate": 8.5e-5,
            "max_steps": 34,
            "seq_length": 2048,
            "packing": False,
            "alignment_stage": "none",
            "notes": "More context, less throughput margin.",
        },
        {
            "run_name": "l4_alignment_ready",
            "hardware": "L4",
            "rank": 16,
            "learning_rate": 8.0e-5,
            "max_steps": 40,
            "seq_length": 2048,
            "packing": True,
            "alignment_stage": "dpo",
            "notes": "Slightly slower SFT in exchange for a cleaner alignment handoff.",
        },
    ]
)

display(run_grid)

## Step 7: Estimate budget and shortlist practical runs

Even a capstone should stay honest about Colab limits, so we attach a lightweight cost model before training anything.

In [ ]:
def estimate_vram_gb(run_row):
    base = 11.2 if run_row["hardware"] == "T4" else 13.5
    rank_cost = 0.05 * run_row["rank"]
    seq_cost = 0.0022 * run_row["seq_length"]
    packing_discount = -0.7 if run_row["packing"] else 0.0
    return round(base + rank_cost + seq_cost + packing_discount, 2)

def fits_hardware(run_row):
    limit = 16.0 if run_row["hardware"] == "T4" else 24.0
    return estimate_vram_gb(run_row) <= (limit - 1.0)

run_grid["estimated_vram_gb"] = run_grid.apply(estimate_vram_gb, axis=1)
run_grid["fits_hardware"] = run_grid.apply(fits_hardware, axis=1)
run_grid["train_rows"] = len(sft_splits["train"])
run_grid["validation_rows"] = len(sft_splits["validation"])

eligible_runs = run_grid[run_grid["fits_hardware"]].copy().reset_index(drop=True)
display(run_grid)
print("Eligible runs:", eligible_runs["run_name"].tolist())

## Step 8: Build or load optional preference pairs for alignment

Notebook 20 exported preference pairs when available. If they are missing, we synthesize a small fallback set by lightly degrading the preferred answer.

In [ ]:
def degrade_answer(text):
    stripped = text.strip()
    if "." in stripped:
        stripped = stripped.split(".")[0] + "."
    return "Short answer: " + stripped.replace("and", ",").replace("should", "can")

def load_preference_records():
    version_root = Path("artifacts") / "notebook_20_project_dataset_pipeline" / "versions"
    if version_root.exists():
        version_dirs = sorted(
            [path_obj for path_obj in version_root.iterdir() if path_obj.is_dir()],
            key=lambda path_obj: path_obj.stat().st_mtime,
            reverse=True,
        )
        if version_dirs:
            pref_path = version_dirs[0] / "preference_pairs.jsonl"
            if pref_path.exists():
                return [json.loads(line) for line in pref_path.read_text(encoding="utf-8").splitlines()]

    fallback_rows = []
    for row in sft_splits["train"]:
        assistant_message = row["messages"][-1]["content"]
        fallback_rows.append(
            {
                "example_id": row["example_id"],
                "dataset_version": DATASET_VERSION,
                "prompt": row["messages"][-2]["content"],
                "chosen": assistant_message,
                "rejected": degrade_answer(assistant_message),
                "scenario_family": row["scenario_family"],
                "rubric_tags": row["rubric_tags"],
            }
        )
    return fallback_rows

preference_records = load_preference_records()
preference_dataset = Dataset.from_list(preference_records)
display(pd.DataFrame(preference_records).head())

## Step 9: Create reusable training-argument builders

The run grid should compile into a concrete trainer configuration instead of staying as a planning-only table.

In [ ]:
def build_run_dir(run_name):
    return RUN_ROOT / run_name

def build_training_args(run_row):
    run_dir = build_run_dir(run_row["run_name"])
    run_dir.mkdir(parents=True, exist_ok=True)
    return TrainingArguments(
        output_dir=str(run_dir),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=float(run_row["learning_rate"]),
        warmup_steps=4,
        max_steps=int(run_row["max_steps"]),
        logging_steps=4,
        evaluation_strategy="steps",
        eval_steps=8,
        save_strategy="steps",
        save_steps=8,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        report_to="none",
        seed=21,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    )

training_args_preview = build_training_args(eligible_runs.iloc[0].to_dict())
print(training_args_preview)

## Step 10: Define a trainer factory for the optional live path

This cell does not start training yet. It simply packages the real SFT path behind a small factory function.

In [ ]:
def build_sft_trainer(run_row):
    if model is None or tokenizer is None:
        return None

    args = build_training_args(run_row)
    return SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=sft_splits["train"],
        eval_dataset=sft_splits["validation"],
        dataset_text_field="text",
        max_seq_length=int(run_row["seq_length"]),
        packing=bool(run_row["packing"]),
        args=args,
    )

print("Live trainer available now:", build_sft_trainer(eligible_runs.iloc[0].to_dict()) is not None)

## Step 11: Simulate the comparison runs

The simulator is intentionally simple. It exists to show how SFT trade-offs surface in loss, formatting, safety, and release score before you commit GPU time.

In [ ]:
def clipped(value, low=0.0, high=1.0):
    return max(low, min(high, value))

def simulate_run(run_row):
    rng = random.Random(run_row["run_name"])
    history_rows = []
    best_val_loss = float("inf")
    best_step = 0

    rank_bonus = 0.012 * (run_row["rank"] / 16)
    seq_penalty = 0.02 if run_row["seq_length"] > 1800 and run_row["hardware"] == "T4" else 0.0
    packing_bonus = 0.015 if run_row["packing"] else -0.005

    for step in range(1, int(run_row["max_steps"]) + 1):
        progress = step / float(run_row["max_steps"])
        train_loss = 1.92 - 0.78 * progress - 0.03 * rank_bonus + rng.uniform(-0.035, 0.02)
        val_loss = 1.75 - 0.52 * progress + 0.16 * (progress ** 2) + seq_penalty + rng.uniform(-0.03, 0.03)
        mentor_score = clipped(0.56 + 0.24 * progress + rank_bonus + packing_bonus + rng.uniform(-0.03, 0.03))
        formatting_score = clipped(0.61 + 0.22 * progress + packing_bonus + rng.uniform(-0.025, 0.025))
        safety_score = clipped(0.81 + 0.06 * progress - 0.01 * (run_row["rank"] > 16) + rng.uniform(-0.015, 0.015))
        calibration_score = clipped(0.58 + 0.20 * progress - seq_penalty + rng.uniform(-0.02, 0.02))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_step = step

        history_rows.append(
            {
                "run_name": run_row["run_name"],
                "step": step,
                "train_loss": round(train_loss, 4),
                "val_loss": round(val_loss, 4),
                "mentor_score": round(mentor_score, 4),
                "formatting_score": round(formatting_score, 4),
                "safety_score": round(safety_score, 4),
                "calibration_score": round(calibration_score, 4),
            }
        )

    history_df = pd.DataFrame(history_rows)
    summary = {
        "run_name": run_row["run_name"],
        "hardware": run_row["hardware"],
        "rank": int(run_row["rank"]),
        "seq_length": int(run_row["seq_length"]),
        "best_val_loss": round(float(best_val_loss), 4),
        "best_step": int(best_step),
        "mentor_score": round(float(history_df["mentor_score"].tail(6).mean()), 4),
        "formatting_score": round(float(history_df["formatting_score"].tail(6).mean()), 4),
        "safety_score": round(float(history_df["safety_score"].tail(6).mean()), 4),
        "calibration_score": round(float(history_df["calibration_score"].tail(6).mean()), 4),
    }
    summary["release_score"] = round(
        0.35 * (1.0 - summary["best_val_loss"] / 2.0)
        + 0.25 * summary["mentor_score"]
        + 0.15 * summary["formatting_score"]
        + 0.15 * summary["safety_score"]
        + 0.10 * summary["calibration_score"],
        4,
    )
    return history_df, summary

history_frames = []
run_summaries = []
for row in eligible_runs.to_dict("records"):
    history_df, summary = simulate_run(row)
    history_frames.append(history_df)
    run_summaries.append(summary)

simulated_history_df = pd.concat(history_frames, ignore_index=True)
comparison_df = pd.DataFrame(run_summaries).sort_values("release_score", ascending=False).reset_index(drop=True)
display(comparison_df)

## Step 12: Pick the strongest SFT candidate

The winning run should not only minimize loss. It should also keep formatting, safety, and calibration strong enough for later benchmarking.

In [ ]:
selected_sft_run = comparison_df.iloc[0].to_dict()
runner_up_run = comparison_df.iloc[1].to_dict() if len(comparison_df) > 1 else comparison_df.iloc[0].to_dict()

print("Selected SFT run:", selected_sft_run["run_name"])
print(to_markdown_table(
    [
        {"metric": "best_val_loss", "selected": selected_sft_run["best_val_loss"], "runner_up": runner_up_run["best_val_loss"]},
        {"metric": "mentor_score", "selected": selected_sft_run["mentor_score"], "runner_up": runner_up_run["mentor_score"]},
        {"metric": "safety_score", "selected": selected_sft_run["safety_score"], "runner_up": runner_up_run["safety_score"]},
        {"metric": "release_score", "selected": selected_sft_run["release_score"], "runner_up": runner_up_run["release_score"]},
    ],
    ["metric", "selected", "runner_up"],
))

## Step 13: Plot the learning curves for the top runs

Visual trends make it easier to see whether the winner is genuinely better or merely a noisy summary-table artifact.

In [ ]:
top_run_names = comparison_df["run_name"].head(2).tolist()
plot_df = simulated_history_df[simulated_history_df["run_name"].isin(top_run_names)].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for run_name in top_run_names:
    run_history = plot_df[plot_df["run_name"] == run_name]
    axes[0].plot(run_history["step"], run_history["train_loss"], label=run_name)
    axes[1].plot(run_history["step"], run_history["val_loss"], label=run_name)
    axes[2].plot(run_history["step"], run_history["mentor_score"], label=run_name)

axes[0].set_title("Train loss")
axes[1].set_title("Validation loss")
axes[2].set_title("Mentor score")
for axis in axes:
    axis.set_xlabel("step")
    axis.legend()

plt.tight_layout()
plt.show()

## Step 14: Define checkpoint policy and run manifests

Checkpoints are not just files. They are decision points tied to metrics, rollback, and later export work.

In [ ]:
checkpoint_policy = {
    "save_every_steps": 8,
    "keep_last_n": 2,
    "keep_best_metric": "best_val_loss",
    "ship_candidate_rule": "best validation loss plus no safety regression below 0.84",
}

checkpoint_manifest_rows = []
for row in comparison_df.to_dict("records"):
    checkpoint_manifest_rows.append(
        {
            "run_name": row["run_name"],
            "checkpoint_dir": str(build_run_dir(row["run_name"]) / "checkpoint-best"),
            "best_step": int(row["best_step"]),
            "best_val_loss": row["best_val_loss"],
            "release_score": row["release_score"],
        }
    )

checkpoint_manifest_df = pd.DataFrame(checkpoint_manifest_rows)
display(checkpoint_manifest_df)
print(json.dumps(checkpoint_policy, indent=2))

## Step 15: Optional live SFT training path

By default this cell only records the plan. Flip LOAD_BASE_MODEL and RUN_ACTUAL_SFT when you want a real experimental run on Colab.

In [ ]:
selected_run_row = eligible_runs[eligible_runs["run_name"] == selected_sft_run["run_name"]].iloc[0].to_dict()
selected_run_plan = {
    "dataset_version": DATASET_VERSION,
    "selected_run": selected_sft_run["run_name"],
    "training_args": build_training_args(selected_run_row).to_dict(),
}

plan_path = ARTIFACT_DIR / "selected_run_plan.json"
with open(plan_path, "w", encoding="utf-8") as handle:
    json.dump(selected_run_plan, handle, indent=2)

if RUN_ACTUAL_SFT:
    if model is None or tokenizer is None:
        raise RuntimeError("Set LOAD_BASE_MODEL = True in the setup cell before enabling RUN_ACTUAL_SFT.")

    trainer = build_sft_trainer(selected_run_row)
    live_train_result = trainer.train()
    trainer.save_model(str(build_run_dir(selected_sft_run["run_name"]) / "adapter"))
    print(live_train_result)
else:
    print("Skipping live SFT. Planning artifact saved to", plan_path)

## Step 16: Plan the optional alignment stage

SFT is the default first stage. Preference alignment becomes worthwhile when the pair data is clean enough and the SFT baseline is already stable.

In [ ]:
alignment_recipes = pd.DataFrame(
    [
        {
            "method": "dpo",
            "needs_reference_model": True,
            "recommended_when": "You have clear chosen and rejected pairs with a stable SFT checkpoint.",
            "colab_note": "Best default second stage when pair quality is strong.",
        },
        {
            "method": "orpo",
            "needs_reference_model": False,
            "recommended_when": "You want a lighter preference update without a separate reference model.",
            "colab_note": "Attractive when memory is tight.",
        },
        {
            "method": "kto",
            "needs_reference_model": False,
            "recommended_when": "You mainly have binary good versus bad signal instead of rich pairwise judgments.",
            "colab_note": "Good for sparse preference feedback.",
        },
    ]
)

def recommend_alignment_method():
    preference_rows = len(preference_records)
    if preference_rows >= 12:
        return "dpo"
    if preference_rows >= 6:
        return "orpo"
    return "kto"

recommended_alignment = recommend_alignment_method()
display(alignment_recipes)
print("Recommended alignment method:", recommended_alignment)

## Step 17: Simulate alignment variants on top of the selected SFT run

This gives the capstone a controlled way to compare SFT-only against a few alignment choices before the live benchmark notebook.

In [ ]:
def simulate_alignment_variant(method_name):
    base = dict(selected_sft_run)
    improvements = {
        "sft_only": {"mentor": 0.00, "formatting": 0.00, "safety": 0.00, "calibration": 0.00},
        "dpo": {"mentor": 0.03, "formatting": 0.02, "safety": 0.025, "calibration": 0.03},
        "orpo": {"mentor": 0.02, "formatting": 0.015, "safety": 0.02, "calibration": 0.015},
        "kto": {"mentor": 0.012, "formatting": 0.01, "safety": 0.018, "calibration": 0.012},
    }
    delta = improvements[method_name]
    return {
        "variant": method_name,
        "mentor_score": round(clipped(base["mentor_score"] + delta["mentor"]), 4),
        "formatting_score": round(clipped(base["formatting_score"] + delta["formatting"]), 4),
        "safety_score": round(clipped(base["safety_score"] + delta["safety"]), 4),
        "calibration_score": round(clipped(base["calibration_score"] + delta["calibration"]), 4),
    }

alignment_rows = []
for variant_name in ["sft_only", "dpo", "orpo", "kto"]:
    row = simulate_alignment_variant(variant_name)
    row["total_score"] = round(
        0.35 * row["mentor_score"]
        + 0.20 * row["formatting_score"]
        + 0.25 * row["safety_score"]
        + 0.20 * row["calibration_score"],
        4,
    )
    alignment_rows.append(row)

alignment_df = pd.DataFrame(alignment_rows).sort_values("total_score", ascending=False).reset_index(drop=True)
selected_release_candidate = alignment_df.iloc[0].to_dict()
display(alignment_df)

## Step 18: Build failure buckets for later regression analysis

Scores alone do not explain what went wrong. Failure buckets turn a weak run into actionable debugging targets.

In [ ]:
failure_bucket_rows = []
for row in comparison_df.to_dict("records"):
    failure_bucket_rows.extend(
        [
            {"run_name": row["run_name"], "bucket": "hallucinated_specifics", "count": int(round(max(0, 0.88 - row["calibration_score"]) * 18))},
            {"run_name": row["run_name"], "bucket": "weak_structure", "count": int(round(max(0, 0.86 - row["formatting_score"]) * 16))},
            {"run_name": row["run_name"], "bucket": "safety_boundary_drift", "count": int(round(max(0, 0.90 - row["safety_score"]) * 20))},
            {"run_name": row["run_name"], "bucket": "underexplained_guidance", "count": int(round(max(0, 0.84 - row["mentor_score"]) * 14))},
        ]
    )

failure_bucket_df = pd.DataFrame(failure_bucket_rows)
display(failure_bucket_df.pivot(index="run_name", columns="bucket", values="count"))

## Step 19: Map failures to mitigation moves

The pipeline is more useful when it not only names failures, but also suggests the next experiment.

In [ ]:
mitigation_map = pd.DataFrame(
    [
        {"bucket": "hallucinated_specifics", "mitigation": "Add more uncertainty-calibrated examples and benchmark prompts."},
        {"bucket": "weak_structure", "mitigation": "Increase format-heavy training examples or add preference data that rewards better structure."},
        {"bucket": "safety_boundary_drift", "mitigation": "Add refusal examples, replay safety data, and tighten release gates."},
        {"bucket": "underexplained_guidance", "mitigation": "Add worked examples and rubric slices that reward stepwise tutoring quality."},
    ]
)

bucket_summary = (
    failure_bucket_df.groupby("bucket")["count"]
    .sum()
    .reset_index()
    .merge(mitigation_map, on="bucket", how="left")
    .sort_values("count", ascending=False)
)
display(bucket_summary)

## Step 20: Package the notebook 22 handoff

The benchmark notebook should not need to rediscover training decisions. It should inherit them as a concrete manifest.

In [ ]:
experiment_registry_path = ARTIFACT_DIR / "experiment_registry.csv"
history_path = ARTIFACT_DIR / "simulated_run_history.csv"
checkpoint_manifest_path = ARTIFACT_DIR / "checkpoint_manifest.csv"
failure_bucket_path = ARTIFACT_DIR / "failure_buckets.csv"
alignment_path = ARTIFACT_DIR / "alignment_comparison.csv"
handoff_path = ARTIFACT_DIR / "handoff_to_notebook_22.json"

comparison_df.to_csv(experiment_registry_path, index=False)
simulated_history_df.to_csv(history_path, index=False)
checkpoint_manifest_df.to_csv(checkpoint_manifest_path, index=False)
failure_bucket_df.to_csv(failure_bucket_path, index=False)
alignment_df.to_csv(alignment_path, index=False)

handoff_manifest = {
    "project": PROJECT_NAME,
    "dataset_version": DATASET_VERSION,
    "selected_sft_run": selected_sft_run["run_name"],
    "selected_release_candidate": selected_release_candidate["variant"],
    "recommended_alignment_method": recommended_alignment,
    "adapter_dir": str(build_run_dir(selected_sft_run["run_name"]) / "adapter"),
    "files": {
        "experiment_registry": str(experiment_registry_path),
        "simulated_history": str(history_path),
        "checkpoint_manifest": str(checkpoint_manifest_path),
        "failure_buckets": str(failure_bucket_path),
        "alignment_comparison": str(alignment_path),
    },
    "release_targets": {
        "mentor_score_min": 0.76,
        "formatting_score_min": 0.78,
        "safety_score_min": 0.84,
    },
}

with open(handoff_path, "w", encoding="utf-8") as handle:
    json.dump(handoff_manifest, handle, indent=2)

print("Saved handoff:", handoff_path)

## Step 21: Reload the saved artifacts and validate the handoff contract

This final check ensures the project state is durable before we move into benchmarking and export.

In [ ]:
reloaded_registry = pd.read_csv(experiment_registry_path)
reloaded_history = pd.read_csv(history_path)
reloaded_checkpoint_manifest = pd.read_csv(checkpoint_manifest_path)
reloaded_failure_buckets = pd.read_csv(failure_bucket_path)
reloaded_alignment = pd.read_csv(alignment_path)
reloaded_handoff = json.loads(handoff_path.read_text(encoding="utf-8"))

artifact_checks = pd.DataFrame(
    [
        {"check": "registry_rows", "value": len(reloaded_registry), "expected": len(comparison_df)},
        {"check": "history_rows", "value": len(reloaded_history), "expected": len(simulated_history_df)},
        {"check": "checkpoint_rows", "value": len(reloaded_checkpoint_manifest), "expected": len(checkpoint_manifest_df)},
        {"check": "failure_bucket_rows", "value": len(reloaded_failure_buckets), "expected": len(failure_bucket_df)},
        {"check": "alignment_rows", "value": len(reloaded_alignment), "expected": len(alignment_df)},
        {"check": "handoff_selected_run", "value": reloaded_handoff["selected_sft_run"], "expected": selected_sft_run["run_name"]},
    ]
)
artifact_checks["pass"] = artifact_checks["value"] == artifact_checks["expected"]
display(artifact_checks)

## Recap

You now have a capstone training pipeline with:

- a versioned dataset handoff from notebook 20
- an SFT comparison grid for Colab-friendly QLoRA runs
- optional preference-alignment planning for DPO, ORPO, or KTO
- checkpoint and failure manifests for regression work
- a durable handoff package for notebook 22 benchmarking and export

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** modify a hyperparameter and track its effect on loss curves. Document what changes and why.

**Exercise 2 — Build:** prepare a dataset from a new domain using the techniques shown. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** compare the finetuned model against the base model on 5 test prompts.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the finetuning/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [20 Project Dataset Pipeline](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/20_project_dataset_pipeline.ipynb) | ➡️ **Next:** [22 Project Benchmark And Export](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/22_project_benchmark_and_export.ipynb)